# A3.7 · The unmanaged agent problem

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

Builds on **[A3.6 · Runtime containment levers](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**.

| | |
|---|---|
| Open-source tooling | Falco, osquery |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A2.4 built an inventory of the identities you know about. This lesson is about
the ones you do not.

Registration is voluntary, and voluntary means partial. The agents that will
hurt you are the ones nobody registered: a contractor's script with a service
account, a team's experiment that quietly became load-bearing, a vendor
integration that spawns workers.

So discovery has to run against **behaviour**, not against a registry. The
signal is that software behaves differently from people, in ways that are
measurable without knowing anything about the actor:

- **Regularity.** Humans are irregular. Loops are metronomic. The coefficient of
  variation of inter-arrival times separates them well.
- **Rate.** Sustained multi-action-per-second activity is not typing.
- **Continuity.** Software has no evenings, no lunch, no weekends.

None of these is sufficient alone, and the honest version of this lesson
includes the cases where the heuristic is **wrong** — because a detection you
have not seen fail is one you will over-trust.

## 2 · Demo — score three actors from telemetry alone

No registry, no names that mean anything. Just authentication and action timestamps, which is what you actually have.

In [ ]:
import statistics, time
from dataclasses import dataclass

@dataclass
class Event:
    ts: float; actor: str; action: str

def agent_score(events, actor):
    ev = sorted((e for e in events if e.actor == actor), key=lambda e: e.ts)
    if len(ev) < 3:
        return {"actor": actor, "score": 0.0, "verdict": "insufficient data"}
    gaps = [b.ts - a.ts for a, b in zip(ev, ev[1:])]
    mean = statistics.fmean(gaps)
    cv = (statistics.pstdev(gaps) / mean) if mean else 0.0
    regularity  = max(0.0, 1.0 - min(cv, 1.0))          # metronomic → 1.0
    rate        = len(ev) / max(ev[-1].ts - ev[0].ts, 1e-9)
    rate_signal = min(rate / 5.0, 1.0)                  # ≥5/s is not a person
    span_hours  = (ev[-1].ts - ev[0].ts) / 3600
    continuity  = min(span_hours / 8.0, 1.0)
    score = round(0.5*regularity + 0.3*rate_signal + 0.2*continuity, 3)
    return {"actor": actor, "score": score,
            "verdict": "agent" if score > 0.6 else "human" if score < 0.3 else "unclear",
            "cv": round(cv, 2), "rate_per_s": round(rate, 2),
            "span_h": round(span_hours, 2)}

now = time.time()
events  = [Event(now + i*0.08, "svc-indexer", "read_file") for i in range(400)]
events += [Event(now + t, "dana@corp", "read_file")
           for t in (0, 4, 11, 12, 60, 130, 133, 400, 900, 1500, 3000)]
events += [Event(now + i*1.0, "unknown-token-7f3c", "http_get") for i in range(300)]

for actor in ("svc-indexer", "dana@corp", "unknown-token-7f3c"):
    r = agent_score(events, actor)
    print(f"{actor:22s} score={r['score']:.3f}  {r['verdict']:9s} "
          f"cv={r.get('cv')}  rate={r.get('rate_per_s')}/s  span={r.get('span_h')}h")

## 3 · Where it breaks — both error directions, deliberately

This is the part that makes the detection usable. Here are two actors the heuristic gets wrong, and neither is contrived.

In [ ]:
# A human whose IDE autosaves on a timer looks metronomic.
ide_user = [Event(now + i*2.0, "sam@corp", "write_file") for i in range(200)]
# An agent written politely, with jittered backoff, looks human.
polite = [Event(now + t, "slow-agent", "http_get")
          for t in (0, 7, 19, 44, 90, 210, 480, 900, 1700, 3000, 5000)]

for evs, actor, truth in ((ide_user, "sam@corp", "human"),
                          (polite, "slow-agent", "agent")):
    r = agent_score(evs, actor)
    wrong = r["verdict"] != truth and r["verdict"] != "unclear"
    print(f"{actor:14s} truth={truth:6s} scored={r['verdict']:9s} "
          f"({r['score']:.3f}){'   ← MISCLASSIFIED' if wrong else ''}")

print("\nThe two errors are not symmetric:")
print("  false 'agent' on a human → an investigation, mild cost, self-correcting")
print("  false 'human' on an agent → it stays invisible, which is the whole risk")
print("So tune the threshold DOWN, and accept the investigations.")

## 4 · The control — discovery joined against the registry

The score alone is not the finding. The finding is **an actor that behaves like software and is not in the inventory.**

In [ ]:
REGISTERED = {"svc-indexer": {"owner": "data-platform"},
              "dana@corp": {"owner": "self"},
              "sam@corp":  {"owner": "self"}}

def discover(events, registry, threshold=0.55):
    findings = []
    for actor in sorted({e.actor for e in events}):
        r = agent_score(events, actor)
        if r["score"] < threshold:
            continue
        reg = registry.get(actor)
        findings.append({
            "actor": actor, "score": r["score"],
            "registered": reg is not None,
            "owner": (reg or {}).get("owner"),
            "finding": None if reg else "SHADOW AGENT — behaves like software, "
                                        "not in the inventory"})
    return findings

all_events = events + ide_user + polite
for f in discover(all_events, REGISTERED):
    flag = f["finding"] or f"registered to {f['owner']}"
    print(f"{f['actor']:22s} score={f['score']:.3f}  {flag}")

shadow = [f for f in discover(all_events, REGISTERED) if f["finding"]]
assert shadow, "a first discovery run always finds at least one"
print(f"\n{len(shadow)} shadow agent(s) to triage.")
print("Each one gets an owner and an entry, or it gets revoked. There is no")
print("third option — an unowned identity that behaves like software is either")
print("someone's load-bearing script or someone else's foothold.")

## What you just proved

`svc-indexer` and `unknown-token-7f3c` score as agents; `dana@corp` scores human. The IDE-autosave user is misclassified as an agent and the politely-jittered agent as human, demonstrating both error directions. Discovery joined against the registry reports `unknown-token-7f3c` as a shadow agent.

## Your turn

Run `agent_score` over one day of real authentication events. Every actor above the threshold that is not in your NHI inventory is a finding, and the first run always produces some. Triage them into owned-or-revoked; there is no third bucket.

---

**Next → [A3.8 · Environment separation](https://spbreed.github.io/cyber-commons/lessons/A3.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*